# Goal 1.5 센서 가용성별 계층형 모델 재현

최초 합성데이터로 만든 `oracle/sanity` 모델을 학습 없이 읽고, Watch·Polar·Muse 1/2/3종 조합 일곱 프로파일의 사건·5단계 prediction을 재현합니다.

실제 Neon 정확도와 장비 동기화는 `NOT VERIFIED`입니다.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tarfile

RUN_TRAINING = False
RUN_LOCKED_TEST = False
USE_GPU = False
assert not RUN_TRAINING and not RUN_LOCKED_TEST and not USE_GPU
print({'RUN_TRAINING': RUN_TRAINING, 'RUN_LOCKED_TEST': RUN_LOCKED_TEST, 'USE_GPU': USE_GPU})

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

manifests = sorted(Path('/kaggle/input').rglob('model_manifest.json'))
if len(manifests) != 1:
    raise RuntimeError(f'모델 manifest는 하나여야 합니다: {manifests}')
model_input = manifests[0].parent
model_root = Path('/kaggle/working/availability_model')
if model_root.exists():
    shutil.rmtree(model_root)
model_root.mkdir(parents=True)
shutil.copy2(model_input / 'model_manifest.json', model_root / 'model_manifest.json')
outer = json.loads((model_root / 'model_manifest.json').read_text())
archive = model_input / outer['archive_path']
payload = model_root / 'extracted'
payload.mkdir()
if archive.is_file():
    if sha256_file(archive) != outer['archive_sha256']:
        raise RuntimeError('모델 archive SHA-256 불일치')
    with tarfile.open(archive, 'r:gz') as bundle:
        unsafe = any(member.name.startswith('/') or '..' in Path(member.name).parts for member in bundle.getmembers())
        if unsafe:
            raise RuntimeError('안전하지 않은 archive member')
        bundle.extractall(payload, filter='data')
else:
    expanded = model_input / 'model_payload'
    if not expanded.is_dir():
        raise RuntimeError('Kaggle model archive 또는 확장 payload가 없습니다')
    shutil.copytree(expanded, payload, dirs_exist_ok=True)
    checksums = model_input / 'SHA256SUMS'
    if checksums.is_file() and outer['archive_sha256'] not in checksums.read_text():
        raise RuntimeError('Kaggle readback checksum manifest 불일치')
wheelhouse = sorted((payload / 'wheel').glob('*.whl'))
runtime = Path('/kaggle/working/multisensor_runtime')
runtime.mkdir(exist_ok=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--target', str(runtime), '--no-index', '--no-deps', *map(str, wheelhouse)])
sys.path.insert(0, str(runtime))
print({'model_root': str(model_root), 'profiles': outer['profile_ids'], 'archive_sha256': outer['archive_sha256']})

In [ ]:
import pandas as pd
from multisensor_ml.sensor_availability import (
    AVAILABILITY_PROFILES,
    load_availability_variant,
    predict_availability_variant,
    verify_sensor_availability_package,
)

if archive.is_file():
    verified = verify_sensor_availability_package(model_root)
else:
    payload_manifest = json.loads((payload / 'payload_manifest.json').read_text())
    for relative, expected_hash in payload_manifest['files'].items():
        if sha256_file(payload / relative) != expected_hash:
            raise RuntimeError(f'payload hash 불일치: {relative}')
    verified = outer
results = []
for profile_id in AVAILABILITY_PROFILES:
    profile_root = payload / 'profiles' / profile_id
    package = load_availability_variant(profile_root)
    sample = pd.read_parquet(payload / 'sample' / f'{profile_id}__input.parquet')
    expected = pd.read_parquet(payload / 'sample' / f'{profile_id}__expected.parquet')
    actual = predict_availability_variant(package, sample)
    pd.testing.assert_frame_equal(actual, expected, check_exact=False, rtol=1e-6, atol=1e-6)
    results.append({'profile_id': profile_id, 'rows': len(actual), 'status': 'REPRODUCED'})
results

In [ ]:
receipt = {
    'status': 'REPRODUCED',
    'profiles': results,
    'model_package_sha256': outer['archive_sha256'],
    'locked_test_read': False,
    'run_training': RUN_TRAINING,
    'run_locked_test': RUN_LOCKED_TEST,
    'data_status': 'oracle/sanity',
    'real_data_status': 'NOT VERIFIED',
    'device_synchronization_status': 'NOT_AVAILABLE_TRUTH_ONLY',
}
receipt_path = Path('/kaggle/working/availability_reproduction_receipt.json')
receipt_path.write_text(json.dumps(receipt, ensure_ascii=False, indent=2, sort_keys=True) + '\n')
print(json.dumps(receipt, ensure_ascii=False, sort_keys=True))

## 해석 경계

프로파일 비교는 센서 가용성에 따른 feature-ablation sanity check입니다. Neon의 실제 Watch 데이터는 payload 디코드·clock correction·품질 파생·관찰자 라벨이 완료된 뒤에만 별도 adapter로 평가합니다.